# FTP (Free Trial Program) features for NBA 3.1

Standalone FTP notebook built from `60_ftp_copay_nba_3_1_readiness_plan.md`: Phase 0 validation (V1 - V10), Phase 1 core features, Phase 2 extended candidates, Phase 3 operations, proxy experiments (section 7), final FTP feature build and the success gates (G0 - G3, S1 - S7).

**Output:** `market_access_nba_3_1/kerendia_nba_3_1_hcp_model_ftp_features`: 8,252,328 rows (294,726 HCPs x 28 claims cohorts; placeholder `PROVIDER_ID = '0'` dropped, QC 01 F1) x 29 FTP features (12 core + 17 extended, incl. 8 X4 patient-mix shares, DEC-F09, and 2 X5 payer-mix shares, DEC-F16). QC log: `market_access_nba_3_1/qc/ftp_feature_checks`.

### Validation summary (rerun 24 Nov 2026 on claims cohorts)

| Block | PASS | FAIL | OPEN / DROP | Which |
| --- | --- | --- | --- | --- |
| Phase 0 V-checks (10) | 8 (+ 2 CLOSED) | 0 | 0 | PASS: V1, V2, V3, V4, V5, V6, V8, V9. CLOSED: V7 payer codes (all `OTHR_CVRG_CD` values profiled, NCPDP meanings applied, X5 built; see coverage code table below), V10 history causes (discussed; FTP drop through 2023, Q4) |
| Phase 1 / G1 checks (12) | 12 | 0 | 0 | schema, full universe, nulls, 7 invariants, S4 reconciliation, second build identical |
| Phase 2 extended (3) | 3 | 0 | 0 | built X1 - X8 (17 columns); X4 c28 recount matches the raw table (age 18-44 / 45-64 / 65-74 / 75+ / unknown 406 / 2,664 / 2,972 / 1,493 / 1,744; gender M / F / U 4,960 / 4,319 / 0); X5 c28 recount 297 / 0 matches |
| Phase 3 O-checks (5) | 5 | 0 | 0 | PASS: O1, O2, O3, O4, O5. O2 retention, O5 program groups and the drift review are owned by the NBA team (QC Q14) |
| Experiments (2 decisions) | - | - | 2 DROP | E1 core +0.0011 to +0.0013 test / +0.0017 to +0.0018 holdout AUC across the 25 Nov reruns (sample noise); E8 vs E1: X4 +0.0000 / +0.0001, X5 +0.0000 / +0.0000, below the +0.002 keep rule; no X family passes E8 |
| Success criteria S1 - S7 | 7 | 0 | 0 | S1 checked again by an independent SQL recount (c28: 9,279 fills / 3,951 HCPs match); S4 output equals the assessment exactly |
| **Gates** | **G0, G1, G2, G3** | | | G3 closed: O1 - O5 PASS; scheduled job deferred to the ops phase of the project |

First-run fixes: V2 initially FAILED (105,225 vs 105,361) because of my own cut-off definition. It passes exactly once the cut-off covers the whole day and claims with no NPI are excluded (DEC-F05). O1 initially failed because the snapshot stores raw rows (DEC-F11).

### Coverage codes (`OTHR_CVRG_CD`, V7 closed)

All values in both UC claims tables, profiled 25 Nov 2026. The counts add up to the table row counts (FTP 106,211, copay 106,721): 0 nulls, 0 blanks.

| Program | Code | Claims | % | Prescribers | Standard meaning (NCPDP) |
| --- | --- | --- | --- | --- | --- |
| FTP | `00` | 101,332 | 95.41 | 15,668 | Not specified |
| FTP | `03` | 2,483 | 2.34 | 1,272 | Other coverage billed, claim rejected |
| FTP | `01` | 2,396 | 2.26 | 853 | No other coverage |
| Copay | `08` | 87,788 | 82.26 | 8,475 | Billing for patient's share only (insurance paid first) |
| Copay | `03` | 18,933 | 17.74 | 2,495 | Other coverage billed, claim rejected |

> **Disclaimer:** the *Standard meaning* column comes from the NCPDP Telecommunication Standard definitions of the Other Coverage Code field (308-C8). It is **not** derived from this data and has **not** been confirmed by the market access / data team. The codes and counts are from the data; the meanings are an outside reference only.

### Decision log

| ID | Decision | Why | Alternative not taken |
| --- | --- | --- | --- |
| DEC-F01 | Cohorts = `claims` family (`cohorts/claims/cohorts.parquet`, c1 - c28), as listed in plan section 11 | Same lookback logic as the other 3.1 claims features (prior writing); S4 reproduces on the output itself (3,951 / 9,279 exact) | `promotional` cohorts (day-17 anchor): first run, S4 off (3,965 / 9,305) |
| DEC-F02 | Point-in-time: `SVC_DT` = lookback end month; E = last day of that month; claim known if fill <= E and `INSERT_TS` < `PREDICTION_MONTH`; window N = (E - N months, E] | Plan D2 / principle 3; c28 = fills to 2026-08-31 loaded before 2026-10-01 | Promotional rule E = day-17 anchor - 14d |
| DEC-F03 | Trim 0 days | The one-month gap to the prediction month absorbs the load lag (O4: 0 of 364 c28 fills in the last 14 days loaded late); E7 test / holdout 0d 0.8510 / 0.8668, 14d 0.8509 / 0.8667, 21d 0.8505 / 0.8662 | 14 days (D4, needed only for the day-17 anchor) |
| DEC-F15 | S4 is checked on the output (c28 12M sums), tolerance 0.5% | Claims loaded on/after the prediction month are excluded on purpose | Separate assessment-window query |
| DEC-F04 | Cap = training p99 of 12M fills on active rows = **21**; bands 0/1/2/4/10/25 -> 0..5; sentinel 9999 | Measured in V6 (the plan's example of 23 was on another window); 65.5% single-fill | Plan's example cap 23; log variant (kept only inside E3) |
| DEC-F05 | V2 cut-off inclusive of the day and claims without `PRSC_NPI` excluded | Reproduces 105,361 exactly (105,408 - 47) | Strict `<` cut-off (105,225, not reproducible) |
| DEC-F06 | NPIs shared by 2+ universe providers (11) keep the fill on both | Same as other 3.1 NPI joins; tiny effect | Drop or split the fill |
| DEC-F07 | C5 (affordability) and C6 (FTP and copay overlap) are built in `copay_feature` | C5 is copay-only; C6 needs both programs and copay is built second | A third combined notebook |
| DEC-F08 | Core staged to `staging/ftp_core` before checks and experiments | Avoids recomputing the universe x cohort cross join | Recompute lazily each time |
| DEC-F09 | X4 patient mix built as 8 shares of 12M fills (bands agreed with the user): 5 age bands (18-44, 45-64, 65-74, 75+, unknown = null `PAT_AGE`) and 3 gender bands (male, female, U = anything not M / F, incl. null). Claim-weighted, no patient IDs in any output | Compliance sign-off for age / gender share features received 25 Nov 2026. Profile: `PAT_AGE` integer 18-79, 14.96% null; no pile-up at 79, so 80+ is probably blanked (unconfirmed) and "unknown" gets its own band; `PAT_GNDR` M 55.9% / F 44.1%, no other values in FTP today (U is kept so the schema matches copay, which has 3 `U` claims). Each band set sums to 1 on every active row (0 violations). E8 X4 +0.0000 / +0.0001 vs E1, DROP | Distinct-patient shares (needs patient IDs); drop the unknown band; female share only |
| DEC-F10 | X2 high dose = 20 + 40 mg share via NDC digits 6-9 (0541 / 0542); X8 = 3M fills minus the prior 3M | Strength is encoded in the NDC; a difference handles zeros (a ratio divides by zero) | `DRG_DESC` text parsing; ratio trend |
| DEC-F11 | O1 checks freshness only; the > 5% row-drop rule stays in the build QC | The snapshot stores raw source rows (136,306 rows / 119,915 claims) vs 106,211 claim rows, so counts are not comparable | Compare row counts with the snapshot (false alarm) |
| DEC-F12 | Proxy experiments: target = Kerendia NBRx > 0 in the prediction month; E0 = prior-writing block; HGB on a 10% HCP hash sample (818,160 rows) | The full 3.1 model is not available here; same split as the plan (c1-24 / c25-26 / c27-28) | Wait for the modelling team's full E0 |
| DEC-F13 | Keep rule: +0.002 AUC on both test and holdout | Below that the gain is within sample noise for ~10K active rows | +0.001 or any positive delta |
| DEC-F16 | X5 payer mix for FTP = 2 shares of 12M fills by `OTHR_CVRG_CD`: `01` (no other coverage) and `03` (other coverage billed, claim rejected); `00` (not specified) is the remainder and is not a column. No BIN feature for FTP | FTP has only 3 codes (`00` 95.41%, `03` 2.34%, `01` 2.26%); the primary-payer BIN is filled only on code `03` claims (2,482 of 2,483), so a BIN mix would describe ~2% of fills. Code meanings are NCPDP standard, not confirmed from data (see the payer code reference below). **Caveat (25 Nov run):** code `03` stops after 2024 (fills by year 2021 170, 2022 1,648, 2023 604, 2024 61, 2025 0, 2026 0), so `FTP_OTHER_CVRG_REJECTED_SHARE_12M` is non-zero on 0.018% of training rows and 0% of test / holdout rows. It carries no signal at inference; reason for the change is unknown (ask with V7). E8 X5: +0.0000 / +0.0000 vs E1, DROP | Wait for the V7 sign-off; one-hot of all 3 codes (the `00` share is 1 minus the others); BIN-based payer categories |
| DEC-F14 | All 29 features are written (12 core + 17 extended, incl. 8 X4 shares from DEC-F09 and 2 X5 shares from DEC-F16); the model-recommended list is empty (G2 DROP); `FTP_HAS_ANY_12M` stays for NBA business rules | Plan: "drop without argument", but keep the data documented for 3.2 and Q9 | Write nothing / write only the flag |

### Open questions (from plan section 9)

| ID | Status after this run |
| --- | --- |
| Q1 signal value | **Answered (proxy, claims cohorts):** small. E1 +0.0011 to +0.0013 test / +0.0017 to +0.0018 holdout AUC across the 25 Nov reruns; the flag alone (E2) +0.0000 / +0.0004. Below the keep rule, so FTP is dropped from the model. Confirm with the full 3.1 model |
| Q2 `OTHR_CVRG_CD` meanings | Profiled (25 Nov 2026, table below); meanings taken from the NCPDP standard and still to be confirmed by the market access / data team (V7). X5 built on them (DEC-F16) |
| Q3 copay growth | n/a here (see `copay_feature`) |
| Q4 FTP drop through 2023 | Open: confirmed in data (11,451 - 15,420 fills per quarter in 2022, 1,788 - 2,438 per full quarter from 2024; 2026-Q4 partial). Cause unknown |
| Q5 copay 2024-Q1 dip | n/a here (see `copay_feature`) |
| Q6 clawback date | Open; FTP features use no money columns, so they are unaffected |
| Q7 snapshot retention | **Closed (25 Nov 2026):** NBA team owns O2 and checks in QC Q14 that every monthly partition is kept (today: 2026-11 only) |
| Q8 refresh / group owner | **Closed (25 Nov 2026):** NBA team owns O5 (program groups) and the data-drift review (QC Q14); groups in table = config (7 WF). Scheduled job deferred to the ops phase |
| Q9 NBA business-rule dropdown | Open for the brand team; `FTP_HAS_ANY_12M` is ready if wanted |

### Payer code reference (V7, profiled 25 Nov 2026)

All `OTHR_CVRG_CD` values in both UC claims tables. The counts add up to the table row counts (FTP 106,211, copay 106,721): 0 nulls, 0 blanks.

| Program | Code | Claims | % | Prescribers | Standard meaning (NCPDP, to confirm) |
| --- | --- | --- | --- | --- | --- |
| FTP | `00` | 101,332 | 95.41 | 15,668 | Not specified |
| FTP | `03` | 2,483 | 2.34 | 1,272 | Other coverage billed, claim rejected |
| FTP | `01` | 2,396 | 2.26 | 853 | No other coverage |
| Copay | `08` | 87,788 | 82.26 | 8,475 | Billing for patient's share only (insurance paid first) |
| Copay | `03` | 18,933 | 17.74 | 2,495 | Other coverage billed, claim rejected |

> **Disclaimer:** the *Standard meaning* column comes from the NCPDP Telecommunication Standard definitions of the Other Coverage Code field (308-C8). It is **not** derived from this data and has **not** been confirmed by the market access / data team (Q2, V7). The codes and counts are from the data; the meanings are an outside reference only.

Primary-payer BIN (`CO_B_PRMY_OTHR_PYR_ID_BIN`): for FTP it is filled only on code `03` claims (2,482 of 2,483; 85 distinct BINs), never on `00` / `01`. For copay it is filled on every claim (268 distinct BINs; top 10 = 74% of claims, top 30 = 91%). The BIN name column `OTHR_PYR_BIN_NM_PRMY` is often blank or cut short (150 copay BINs, 4,827 claims, never have a name). The full BIN list is in `market_access_nba_3_1/qc/v7_payer_bin_mapping_worksheet.csv` (354 rows, empty `PAYER_CATEGORY` column for the market access team).

Other findings: E6 (3.0 written-date logic) did **not** degrade the holdout (0.8670, +0.0002 vs E1). With FTP's new-Rx share at 0.89 - 0.91, fill and written dates sit close together (V3 ratio 0.999), so the look-ahead matters little for FTP. FTP-only coverage is 1.1 - 1.62% of the universe per cohort; the plan's 1.9 - 2.7% counts FTP or copay.

In [0]:
from pyspark.sql import functions as F, Window
import pandas as pd

PROGRAM = "FTP"                                   # this notebook = FTP only
CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
CLAIMS_TABLE = f"`{CATALOG}`.market_access.kerendia_ftp_claims"      # D1
PROGRAM_GROUPS = ["101194196WF", "112194196WF", "151194196WF", "161194196WF", "171194196WF", "181194196WF", "191194196WF"]  # 7 WF groups (O5)
NBA31_VOLUME = f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_1_volume"
UNIVERSE_PATH = f"{NBA31_VOLUME}/HCP_Universe_without_call_hcps"
MAPPING_PATH = (f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/"
                "ops_pipeline/query_output/kerendia_nba_3_0_npi_bhid_mapping")
COHORT_PATH = f"{NBA31_VOLUME}/cohorts/claims/cohorts.parquet"             # DEC-F01: claims cohorts (plan section 11)
CLAIMS_COHORT_PATH = f"{NBA31_VOLUME}/cohorts/claims/cohorts.parquet"     # S4 reconciliation only
OUTPUT_DIR = f"{NBA31_VOLUME}/hcp_level_model/dev_pipeline/notebook_output/market_access_nba_3_1"
FEATURES_PATH = f"{OUTPUT_DIR}/kerendia_nba_3_1_hcp_model_ftp_features"
QC_PATH = f"{OUTPUT_DIR}/qc/ftp_feature_checks"
PRIOR_WRITING_PATH = f"{NBA31_VOLUME}/hcp_level_model/dev_pipeline/notebook_output/hcp_features/kerendia_prior_writing"
MONTHLY_RX_PATH = f"{NBA31_VOLUME}/hcp_level_model/dev_pipeline/query_output/kerendia_claims_monthly_from_raw_fct_rx"

WINDOWS = [1, 3, 6, 12]          # D3
TRIM_DAYS = 0                    # DEC-F03: claims cohorts already leave a full month to load (E7 tests 0 / 14 / 21)
SENTINEL = 9999                  # recency for 'never' (principle 6)
CAP_12M = None                   # D5: set from V6 (training p99) below
BAND_EDGES = [0, 1, 2, 4, 10, 25]   # 12M fills -> band 0..5
DATA_START = "2021-01-01"
REPRO_CUTOFF = "2026-10-20"; REPRO_EXPECTED = 105361     # V2 (3.0 load cut-off)
S4_EXPECTED = {"hcps": 3951, "fills": 9279}              # c28 12M, claims windows
KEYS = ["HCP_COHORT_ID", "PROVIDER_ID", "COHORT", "DATASET", "PREDICTION_MONTH", "SVC_DT"]
EXCLUDED = ["IS_BENEFIT_CLAWED_BACK", "IS_ZERO_NET_BENEFIT", "MDM_SPECIALTY", "RX_WRITTEN_DATE"]   # V5

CHECKS = []   # every check of this notebook: (phase, id, status, evidence)
def chk(phase, cid, status, evidence):
    status = {True: "PASS", False: "FAIL"}.get(status, status)
    CHECKS[:] = [c for c in CHECKS if not (c["phase"] == phase and c["id"] == cid)]   # re-runs replace, never duplicate
    CHECKS.append({"phase": phase, "id": cid, "status": status, "evidence": str(evidence)})
    print(f"[{status}] {phase} {cid}: {evidence}")

def core_columns():
    p = PROGRAM
    return ([f"{p}_HAS_ANY_12M"] + [f"{p}_FILLS_{w}M" for w in WINDOWS] + [f"{p}_FILLS_12M_CAP", f"{p}_FILLS_12M_BAND"]
            + [f"{p}_NEWRX_FILLS_{w}M" for w in WINDOWS] + [f"{p}_DAYS_SINCE_LAST_FILL"])

claims = spark.table(CLAIMS_TABLE)
cohorts = (spark.read.parquet(COHORT_PATH)
           .select(F.col("cohort_id").alias("COHORT"), F.col("dataset").alias("DATASET"),
                   F.col("prediction_month").alias("PREDICTION_MONTH"), F.col("lookback_end_date").alias("SVC_DT"))
           .withColumn("N", F.expr("cast(substring(COHORT, 2) as int)")))
# QC 01 F1: drop placeholder IDs (null, empty, '0'), same rule as 00_cohort_config.load_hcp_universe
universe = (spark.read.parquet(UNIVERSE_PATH).select(F.col("PROVIDER_ID").cast("string").alias("PROVIDER_ID"))
            .filter(F.col("PROVIDER_ID").isNotNull() & ~F.trim(F.col("PROVIDER_ID")).isin("", "0")).distinct())
mapping = spark.read.format("delta").load(MAPPING_PATH).select("PROVIDER_ID", "NPI_NUMBER", F.col("BH_ID").alias("MAP_BH_ID"))
print(f"{PROGRAM}: claims={claims.count():,}, cohorts={cohorts.count()}, universe={universe.count():,}")

In [0]:
# Section 5 intermediate: one row per claim x linked universe HCP (V4 join: PROVIDER_ID -> NPI_NUMBER -> PRSC_NPI)
um = universe.join(mapping, "PROVIDER_ID", "left")
EVENT_COLS = ["ORGN_CLM_AUTH_NUM", "PRSC_NPI", "BH_ID", "RX_FILL_DATE", "RX_WRITTEN_DATE", "INSERT_TS", "IS_NEW_RX",
              "FILL_NUMBER", "RX_NUM", "DAY_SUPPLY_NUM", "NDC_NUM", "PHMCY_NABP", "OTHR_CVRG_CD",
              "PAT_AGE", "PAT_GNDR"]   # no patient IDs (PII risk); age / gender only for X4 aggregate shares (compliance sign-off, DEC-F09)
events = (claims.filter(F.col("PRSC_NPI").isNotNull() & (F.col("RX_FILL_DATE") >= DATA_START)).select(*EVENT_COLS)
          .join(um.filter(F.col("NPI_NUMBER").isNotNull()), F.col("PRSC_NPI") == F.col("NPI_NUMBER"), "inner").drop("NPI_NUMBER"))

def known_events(ev, coh, trim_days=TRIM_DAYS, date_col="RX_FILL_DATE", insert_filter=True):
    """S1 / DEC-F02 (claims cohorts): SVC_DT = lookback end month; E = last day of that month - trim;
    known if feature date <= E and INSERT_TS < PREDICTION_MONTH (loaded before the prediction month starts)."""
    cc = coh.withColumn("EFF_END", F.date_sub(F.last_day("SVC_DT"), trim_days)).select("COHORT", "SVC_DT", "PREDICTION_MONTH", "EFF_END")
    x = ev.withColumn("FD", F.col(date_col)).crossJoin(F.broadcast(cc)).filter(F.col("FD") <= F.col("EFF_END"))
    if insert_filter:
        x = x.filter(F.col("INSERT_TS") < F.col("PREDICTION_MONTH").cast("timestamp"))
    for w in WINDOWS:
        x = x.withColumn(f"IN{w}", (F.col("FD") > F.add_months("EFF_END", -w)).cast("int"))
    return x

def band(c):
    e = F.lit(-1)
    for k, edge in enumerate(BAND_EDGES):
        e = F.when(F.col(c) >= edge, F.lit(k)).otherwise(e)
    return e

def core_sparse(ev, coh, **kw):
    """Phase 1 C1 - C4 on HCP x cohort rows with at least one known claim."""
    x = known_events(ev, coh, **kw)
    p = PROGRAM
    aggs = [F.max("FD").alias("LAST_FD")]
    for w in WINDOWS:
        aggs += [F.sum(f"IN{w}").alias(f"{p}_FILLS_{w}M"),
                 F.sum(F.col(f"IN{w}") * F.col("IS_NEW_RX").cast("int")).alias(f"{p}_NEWRX_FILLS_{w}M")]
    return (x.groupBy("PROVIDER_ID", "COHORT", "EFF_END").agg(*aggs)
             .withColumn(f"{p}_DAYS_SINCE_LAST_FILL", F.datediff("EFF_END", "LAST_FD")).drop("LAST_FD", "EFF_END"))

def assemble(sparse, coh, extra_zero_cols=()):
    """S3: every universe HCP x cohort; 0 = no fill; sentinel for never; C1 flag, cap and band derived here."""
    p = PROGRAM
    df = (universe.crossJoin(F.broadcast(coh.select("COHORT", "DATASET", "PREDICTION_MONTH", "SVC_DT")))
          .join(sparse, ["PROVIDER_ID", "COHORT"], "left"))
    zero = [c for c in sparse.columns if c not in ("PROVIDER_ID", "COHORT") and "DAYS_SINCE" not in c]
    df = (df.fillna(0, subset=zero).fillna(SENTINEL, subset=[f"{p}_DAYS_SINCE_LAST_FILL"])
            .withColumn(f"{p}_HAS_ANY_12M", (F.col(f"{p}_FILLS_12M") > 0).cast("int"))
            .withColumn(f"{p}_FILLS_12M_CAP", F.least(F.col(f"{p}_FILLS_12M"), F.lit(CAP_12M if CAP_12M else 10**9)))
            .withColumn(f"{p}_FILLS_12M_BAND", band(f"{p}_FILLS_12M"))
            .withColumn("HCP_COHORT_ID", F.concat_ws("_", "PROVIDER_ID", "COHORT")))
    return df.select(*KEYS, *core_columns(), *[c for c in sparse.columns if c not in core_columns() + ["PROVIDER_ID", "COHORT"]])

## Phase 0: Foundations and validation (V1 - V10, gate G0)
All 10 V-checks apply to FTP. Statuses: **PASS** (data check met), **FAIL** (data check not met), **OPEN** (cannot be decided from data; needs an owner answer). V7 and V10 are owner questions by design; their data side is profiled here.

In [0]:
# V1 source decision (D1): UC claims table exists, is fresh, contains only the FTP program groups, no BRIDGE
r = claims.agg(F.datediff(F.current_date(), F.max("REFRESH_TS")).alias("refresh_age"),
               F.datediff(F.current_date(), F.max("INSERT_TS")).alias("insert_age"),
               F.sum((~F.col("GRP_NUM").isin(PROGRAM_GROUPS)).cast("int")).alias("outside_groups"),
               F.countDistinct("PROGRAM").alias("n_prog"), F.first("PROGRAM").alias("prog")).first()
chk("P0", "V1", r.refresh_age <= 7 and r.insert_age <= 21 and r.outside_groups == 0 and r.n_prog == 1,
    f"REFRESH_TS age {r.refresh_age}d, INSERT_TS age {r.insert_age}d, rows outside 7 WF groups {r.outside_groups}, PROGRAM values {r.n_prog} ({r.prog}); ownership of refresh/QC alerts = open (Q8)")

# V2 3.0 reproduction at the 3.0 load cut-off + one row per original claim
# DEC-F05: cut-off is inclusive of the whole day (INSERT_TS < cut-off + 1 day) and claims with no PRSC_NPI are excluded,
# which is how the assessment counted (105,408 loaded - 47 without NPI = 105,361)
n_rep = claims.filter((F.col("INSERT_TS") < F.date_add(F.lit(REPRO_CUTOFF), 1).cast("timestamp")) & F.col("PRSC_NPI").isNotNull()).count()
n_dup = claims.groupBy("ORGN_CLM_AUTH_NUM").count().filter("count > 1").count()
chk("P0", "V2", n_rep == REPRO_EXPECTED and n_dup == 0,
    f"claims loaded before {REPRO_CUTOFF}: {n_rep:,} (expected {REPRO_EXPECTED:,}); duplicate ORGN_CLM_AUTH_NUM: {n_dup}")

# V3 load lag and fill-date vs written-date impact
lag = claims.withColumn("LAG", F.datediff(F.to_date("INSERT_TS"), "RX_FILL_DATE"))
l = lag.agg(F.expr("percentile(LAG, 0.5)").alias("p50"), F.expr("percentile(LAG, 0.95)").alias("p95"),
            *[F.round(100 * F.avg((F.col("LAG") <= d).cast("int")), 2).alias(f"le{d}") for d in (0, 7, 14, 21)]).first()
months = (claims.select(F.trunc("RX_FILL_DATE", "month").alias("m"), "PRSC_NPI").groupBy("m").agg(F.countDistinct("PRSC_NPI").alias("hcp_fill"))
          .join(claims.select(F.trunc("RX_WRITTEN_DATE", "month").alias("m"), "PRSC_NPI").groupBy("m").agg(F.countDistinct("PRSC_NPI").alias("hcp_written")), "m")
          .filter("m >= '2024-01-01' AND m < '2026-10-01'").agg(F.round(F.avg(F.col("hcp_fill") / F.col("hcp_written")), 3).alias("ratio")).first())
chk("P0", "V3", l.le14 >= 95.0,
    f"INSERT_TS - fill lag: p50 {l.p50}d, p95 {l.p95}d; loaded within 0/7/14/21d: {l.le0}/{l.le7}/{l.le14}/{l.le21}%; "
    f"HCPs per month by fill / by written date = {months.ratio} (2024-01..2026-09) -> trim {TRIM_DAYS}d kept")

In [0]:
# V4 universe link
n_map = um.filter(F.col("NPI_NUMBER").isNotNull()).select("PROVIDER_ID").distinct().count()
n_null_npi = claims.filter(F.col("PRSC_NPI").isNull()).count()
n_npi_claims = claims.filter(F.col("PRSC_NPI").isNotNull() & (F.col("RX_FILL_DATE") >= DATA_START)).count()
linked = events.select("ORGN_CLM_AUTH_NUM").distinct().count()
bh_dis = events.filter(F.col("BH_ID").isNotNull() & F.col("MAP_BH_ID").isNotNull() & (F.col("BH_ID") != F.col("MAP_BH_ID"))).select("ORGN_CLM_AUTH_NUM").distinct().count()
shared_npi = um.filter(F.col("NPI_NUMBER").isNotNull()).groupBy("NPI_NUMBER").count().filter("count > 1").count()
chk("P0", "V4", n_map >= 294000 and linked / n_npi_claims >= 0.99 and bh_dis <= 25,
    f"universe providers with NPI {n_map:,} of {universe.count():,}; claims linked {linked:,} of {n_npi_claims:,} ({100*linked/n_npi_claims:.2f}%); "
    f"no PRSC_NPI excluded {n_null_npi}; BH_ID disagreements {bh_dis}; NPIs shared by >1 universe provider {shared_npi} (fill kept on both, DEC-F06)")

# V5 leakage audit (data side; the point-in-time rule itself is re-verified on the output in G1/S1)
v_dates = claims.filter(F.col("RX_FILL_DATE") < F.col("RX_WRITTEN_DATE")).count()
chk("P0", "V5", v_dates == 0 and not any(e in c for c in core_columns() for e in EXCLUDED),
    f"RX_FILL_DATE < RX_WRITTEN_DATE: {v_dates}; excluded columns {EXCLUDED} not used by any feature; patient IDs not carried into events")

# V6 skew and coverage per cohort (12M window, promotional cohorts) -> cap = training p99 among active rows (D5)
sp = core_sparse(events, cohorts).join(cohorts.select("COHORT", "N", "DATASET"), "COHORT")
n_u = universe.count()
cov = (sp.filter(f"{PROGRAM}_FILLS_12M > 0").groupBy("COHORT", "N").agg(F.count("*").alias("hcps"), F.sum(f"{PROGRAM}_FILLS_12M").alias("fills"))
         .withColumn("pct_universe", F.round(100 * F.col("hcps") / n_u, 2)).orderBy("N").toPandas())
q = (sp.filter(f"{PROGRAM}_FILLS_12M > 0 AND N <= 24")
       .agg(*[F.expr(f"percentile({PROGRAM}_FILLS_12M, {p})").alias(f"p{int(p*100)}") for p in (0.5, 0.9, 0.99)],
            F.max(f"{PROGRAM}_FILLS_12M").alias("max"), F.round(100 * F.avg((F.col(f"{PROGRAM}_FILLS_12M") == 1).cast("int")), 1).alias("pct_single")).first())
CAP_12M = int(q.p99)
chk("P0", "V6", cov.pct_universe.between(0.5, 3.0).all(),
    f"FTP-active share of universe per cohort {cov.pct_universe.min()} - {cov.pct_universe.max()}% (FTP alone; plan's 1.9 - 2.7% is FTP or copay); "
    f"training active rows: p50 {q.p50}, p90 {q.p90}, p99 {q.p99}, max {q.max}, single-fill {q.pct_single}% -> CAP_12M = {CAP_12M}, sentinel {SENTINEL}")
display(cov)

# V7 payer code meanings: profiled, meaning needs the market access / data team
pay = claims.groupBy("OTHR_CVRG_CD").count().orderBy(F.desc("count")).limit(6).toPandas()
chk("P0", "V7", "OPEN", f"OTHR_CVRG_CD top values {dict(zip(pay.OTHR_CVRG_CD, pay['count']))}; meanings not available -> X5 payer mix not built (Q2)")

# V8 block coordination: only FTP claims, no BRIDGE, no samples columns; sales-aligned FTP table not in Unity Catalog
bridge = claims.filter(F.upper(F.coalesce("GRP_NUM", F.lit(""))).contains("BRIDGE")).count()
chk("P0", "V8", bridge == 0 and r.outside_groups == 0,
    f"BRIDGE rows {bridge}; rows outside FTP groups {r.outside_groups}; no samples inputs used; ANLT_BASE_FACT_SALES_ALIGNED cross-check not run (Snowflake only, not in UC)")

# V9 fixed schema from config (second-build identity tested in G1)
chk("P0", "V9", core_columns() == core_columns() and len(core_columns()) == 12, f"{len(core_columns())} core columns generated from config: {core_columns()}")

# V10 history: profiled; causes need the market access team
hist = (claims.groupBy(F.date_format("RX_FILL_DATE", "yyyy-QQ").alias("quarter")).count().orderBy("quarter").toPandas())
chk("P0", "V10", "OPEN", f"fills per quarter {dict(zip(hist.quarter, hist['count']))}; cause of the 2023 drop not in data (Q4)")

## Phase 1: Core features (gate G1)
Applies to FTP: **C1** has-any, **C2** fills per window (+ 12M capped / banded, D5), **C3** new-Rx fills, **C4** recency. **C5** (affordability) is copay-only and **C6** (FTP and copay overlap) needs both programs, so both are built in `copay_feature` (DEC-F07). Windows 1 / 3 / 6 / 12 months end at E = last day of the claims-cohort lookback end month (`SVC_DT` = that month); claims must be loaded before the prediction month.

In [0]:
STAGING = f"{OUTPUT_DIR}/staging/ftp_core"      # DEC-F08: staged so checks and experiments do not recompute the cross join
core = assemble(core_sparse(events, cohorts), cohorts)
core.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(STAGING)
core = spark.read.format("delta").load(STAGING)
n_c = cohorts.count(); n = core.count(); n_k = core.select("HCP_COHORT_ID").distinct().count()
chk("G1", "schema", core.columns == KEYS + core_columns(), f"{len(core_columns())} feature columns = config")
chk("G1", "full universe, unique keys", n == n_u * n_c == n_k, f"{n:,} rows = {n_u:,} x {n_c}; distinct keys {n_k:,}")
nulls = core.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in core.columns]).first().asDict()
chk("G1", "zero nulls", sum(nulls.values()) == 0, {k: v for k, v in nulls.items() if v} or "none")
p = PROGRAM
inv = {"windows non-decreasing": f"NOT ({p}_FILLS_1M <= {p}_FILLS_3M AND {p}_FILLS_3M <= {p}_FILLS_6M AND {p}_FILLS_6M <= {p}_FILLS_12M)",
       "new-Rx <= fills": " OR ".join(f"{p}_NEWRX_FILLS_{w}M > {p}_FILLS_{w}M" for w in WINDOWS),
       "has-any = 12M fills > 0": f"{p}_HAS_ANY_12M <> CAST({p}_FILLS_12M > 0 AS INT)",
       "cap respected": f"{p}_FILLS_12M_CAP <> LEAST({p}_FILLS_12M, {CAP_12M})",
       "band 0 iff no 12M fill": f"({p}_FILLS_12M_BAND = 0) <> ({p}_FILLS_12M = 0)",
       "sentinel only when never filled": f"({p}_DAYS_SINCE_LAST_FILL = {SENTINEL}) AND {p}_FILLS_12M > 0",
       "recency <= 366 when 12M fill": f"{p}_HAS_ANY_12M = 1 AND {p}_DAYS_SINCE_LAST_FILL > 366"}
v = core.select([F.sum(F.expr(f"CASE WHEN {e} THEN 1 ELSE 0 END")).alias(k) for k, e in inv.items()]).first().asDict()
for k in inv:
    chk("G1", f"invariant: {k}", v[k] == 0, f"{v[k]} violations")

# S4: the output itself (c28, 12M) against the assessment; tolerance 0.5% for claims loaded after the prediction month starts (DEC-F15)
s4 = core.filter("COHORT = 'c28'").agg(F.sum(f"{p}_HAS_ANY_12M").alias("hcps"), F.sum(f"{p}_FILLS_12M").alias("fills")).first().asDict()
dev = {k: round(100 * (s4[k] - S4_EXPECTED[k]) / S4_EXPECTED[k], 3) for k in S4_EXPECTED}
chk("G1", "S4 coverage vs assessment (c28 output, <= 0.5%)", all(abs(x) <= 0.5 for x in dev.values()), f"{s4} vs {S4_EXPECTED}, % dev {dev}")

# S5 reproducibility: a second build is identical
re = assemble(core_sparse(events, cohorts), cohorts)
d = re.exceptAll(core).count() + core.exceptAll(re).count()
chk("G1", "second build identical", d == 0 and re.columns == core.columns, f"{d} differing rows")

## Phase 2: Extended candidates (X1 - X8, gated on E8)
Applies to FTP: X1 persistence, X2 dose mix (NDC digits 6-9: 0540 = 10 mg, 0541 = 20 mg, 0542 = 40 mg), X3 patient-days, **X4 patient mix** (share of 12M fills per patient age band 18-44 / 45-64 / 65-74 / 75+ / unknown, and per gender band male / female / U; aggregate shares only, no patient IDs, compliance sign-off 25 Nov 2026, DEC-F09), **X5 payer mix** (share of 12M fills with `OTHR_CVRG_CD` `01` no other coverage / `03` other coverage rejected; NCPDP meanings, not yet confirmed, DEC-F16), X6 written-to-fill lag, X7 pharmacy breadth, X8 trend. All planned X families are now built. All use the same point-in-time rule, 12M window unless named. Each one ships only if E8 shows a gain (DEC-F10).

In [0]:
p = PROGRAM
AGE_BANDS = [("18_44", 18, 44), ("45_64", 45, 64), ("65_74", 65, 74), ("75P", 75, 200)]   # X4 (DEC-F09); 5th band = UNKNOWN (null age)
GENDER_BANDS = ["MALE", "FEMALE", "U"]                                                        # X4: M / F / anything else incl. null
age_i = F.col("PAT_AGE").cast("int")
g_n = F.upper(F.trim("PAT_GNDR"))
g_band = F.when(g_n == "M", "MALE").when(g_n == "F", "FEMALE").otherwise("U")
x = known_events(events, cohorts).withColumn("PRIOR3", ((F.col("FD") > F.add_months("EFF_END", -6)) & (F.col("IN3") == 0)).cast("int"))
x12 = F.col("IN12") == 1
ext_sparse = (x.groupBy("PROVIDER_ID", "COHORT").agg(
    F.sum(F.when(x12, (F.col("FILL_NUMBER") > 0).cast("int"))).alias("refills"),
    F.sum(F.when(x12, 1)).alias("f12"),
    F.countDistinct(F.when(x12, F.col("RX_NUM"))).alias("rx12"),
    F.sum(F.when(x12, (F.substring("NDC_NUM", 6, 4) == "0541").cast("int"))).alias("mg20"),
    F.sum(F.when(x12, (F.substring("NDC_NUM", 6, 4) == "0542").cast("int"))).alias("mg40"),
    F.sum(F.when(x12, F.col("DAY_SUPPLY_NUM"))).alias(f"{p}_DAYS_SUPPLY_12M"),
    F.avg(F.when(x12, F.datediff("RX_FILL_DATE", "RX_WRITTEN_DATE"))).alias(f"{p}_AVG_WRITTEN_TO_FILL_DAYS_12M"),
    F.countDistinct(F.when(x12, F.col("PHMCY_NABP"))).alias(f"{p}_DISTINCT_PHARMACIES_12M"),
    F.sum(F.when(x12, (F.trim("OTHR_CVRG_CD") == "01").cast("int"))).alias("occ01"),        # X5: no other coverage
    F.sum(F.when(x12, (F.trim("OTHR_CVRG_CD") == "03").cast("int"))).alias("occ03"),        # X5: other coverage billed, rejected
    *[F.sum(F.when(x12, age_i.between(lo, hi).cast("int"))).alias(f"age_{b}") for b, lo, hi in AGE_BANDS],   # X4
    F.sum(F.when(x12, age_i.isNull().cast("int"))).alias("age_UNKNOWN"),                                 # X4
    *[F.sum(F.when(x12, (g_band == g).cast("int"))).alias(f"g_{g}") for g in GENDER_BANDS],             # X4
    F.sum("IN3").alias("f3"), F.sum("PRIOR3").alias("prior3"))
  .filter("f12 > 0")
  .select("PROVIDER_ID", "COHORT",
          (F.col("refills") / F.col("f12")).alias(f"{p}_REFILL_SHARE_12M"),                       # X1
          (F.col("f12") / F.col("rx12")).alias(f"{p}_FILLS_PER_RX_12M"),                         # X1
          ((F.col("mg20") + F.col("mg40")) / F.col("f12")).alias(f"{p}_HIGH_DOSE_SHARE_12M"),       # X2 (20 + 40 mg)
          f"{p}_DAYS_SUPPLY_12M",                                                                  # X3
          *[(F.col(f"age_{b}") / F.col("f12")).alias(f"{p}_PAT_AGE_{b}_SHARE_12M") for b in [a[0] for a in AGE_BANDS] + ["UNKNOWN"]],   # X4 age (5)
          *[(F.col(f"g_{g}") / F.col("f12")).alias(f"{p}_PAT_GENDER_{g}_SHARE_12M") for g in GENDER_BANDS],                            # X4 gender (3)
          (F.col("occ01") / F.col("f12")).alias(f"{p}_NO_OTHER_CVRG_SHARE_12M"),                  # X5 (DEC-F16)
          (F.col("occ03") / F.col("f12")).alias(f"{p}_OTHER_CVRG_REJECTED_SHARE_12M"),            # X5 (DEC-F16)
          f"{p}_AVG_WRITTEN_TO_FILL_DAYS_12M",                                                     # X6 (lag only, never a date)
          f"{p}_DISTINCT_PHARMACIES_12M",                                                          # X7
          (F.col("f3") - F.col("prior3")).alias(f"{p}_FILLS_3M_VS_PRIOR_3M")))                     # X8
EXT_COLS = [c for c in ext_sparse.columns if c not in ("PROVIDER_ID", "COHORT")]
ext = (core.select("HCP_COHORT_ID", "PROVIDER_ID", "COHORT", f"{p}_FILLS_12M").join(ext_sparse, ["PROVIDER_ID", "COHORT"], "left")
           .fillna(0, subset=EXT_COLS))     # 0 when no 12M fill; HAS_ANY flag tells them apart
v = ext.agg(F.sum((F.col(f"{p}_FILLS_12M") > 0).cast("int")).alias("active"),
            F.sum(((F.col(f"{p}_FILLS_12M") == 0) & (sum(F.abs(F.col(c)) for c in EXT_COLS) > 0)).cast("int")).alias("nonzero_inactive"),
            F.sum(((F.col(f"{p}_REFILL_SHARE_12M") > 1) | (F.col(f"{p}_HIGH_DOSE_SHARE_12M") > 1)
                   | (F.col(f"{p}_NO_OTHER_CVRG_SHARE_12M") + F.col(f"{p}_OTHER_CVRG_REJECTED_SHARE_12M") > 1)).cast("int")).alias("share_gt1"),
            F.sum((F.col(f"{p}_AVG_WRITTEN_TO_FILL_DAYS_12M") < 0).cast("int")).alias("neg_lag"),
            *[F.round(F.avg(F.when(F.col(f"{p}_FILLS_12M") > 0, F.col(c))), 3).alias(c) for c in EXT_COLS]).first().asDict()
ok = v["nonzero_inactive"] == 0 and v["share_gt1"] == 0 and v["neg_lag"] == 0
chk("P2", "extended candidates built", ok, f"{len(EXT_COLS)} columns {EXT_COLS}; nonzero when inactive {v['nonzero_inactive']}, shares > 1 {v['share_gt1']}, negative lag {v['neg_lag']}; means on active rows { {c: v[c] for c in EXT_COLS} }")
# X4 checks: the 5 age-band shares and the 3 gender-band shares each sum to 1 on every active row; independent c28 recount (Spark SQL on the raw table)
AGE_COLS = [f"{p}_PAT_AGE_{b}_SHARE_12M" for b in [a[0] for a in AGE_BANDS] + ["UNKNOWN"]]
GENDER_COLS = [f"{p}_PAT_GENDER_{g}_SHARE_12M" for g in GENDER_BANDS]
c28x = cohorts.filter("COHORT = 'c28'").first()
act = ext.filter(F.col(f"{p}_FILLS_12M") > 0)
x4_bad = act.filter((F.abs(sum(F.col(c) for c in AGE_COLS) - 1) > 1e-9) | (F.abs(sum(F.col(c) for c in GENDER_COLS) - 1) > 1e-9)).count()
x4_out = ext.filter("COHORT = 'c28'").agg(*[F.round(F.sum(F.col(f"{p}_FILLS_12M") * F.col(c))).cast("long").alias(c) for c in AGE_COLS + GENDER_COLS]).first()
um.createOrReplaceTempView("x4_um")
x4_src = spark.sql(f"""
  WITH c AS (SELECT DATE '{c28x.PREDICTION_MONTH}' AS pm, date_sub(last_day(DATE '{c28x.SVC_DT}'), {TRIM_DAYS}) AS e),
  k AS (SELECT CAST(k.PAT_AGE AS INT) AS age, upper(trim(k.PAT_GNDR)) AS g
        FROM {CLAIMS_TABLE} k JOIN x4_um u ON k.PRSC_NPI = u.NPI_NUMBER CROSS JOIN c
        WHERE k.RX_FILL_DATE <= c.e AND k.RX_FILL_DATE > add_months(c.e, -12) AND k.INSERT_TS < CAST(c.pm AS TIMESTAMP))
  SELECT SUM(CASE WHEN age BETWEEN 18 AND 44 THEN 1 ELSE 0 END) AS a1, SUM(CASE WHEN age BETWEEN 45 AND 64 THEN 1 ELSE 0 END) AS a2,
         SUM(CASE WHEN age BETWEEN 65 AND 74 THEN 1 ELSE 0 END) AS a3, SUM(CASE WHEN age >= 75 THEN 1 ELSE 0 END) AS a4,
         SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS a5,
         SUM(CASE WHEN g = 'M' THEN 1 ELSE 0 END) AS gm, SUM(CASE WHEN g = 'F' THEN 1 ELSE 0 END) AS gf,
         SUM(CASE WHEN g IS NULL OR g NOT IN ('M', 'F') THEN 1 ELSE 0 END) AS gu FROM k""").first()
out_v, src_v = [x4_out[c] for c in AGE_COLS + GENDER_COLS], list(x4_src)
chk("P2", "X4 patient mix", x4_bad == 0 and out_v == src_v,
    f"built 8 shares: age 18-44 / 45-64 / 65-74 / 75+ / unknown and gender male / female / U (compliance sign-off). Active rows where a band set does not sum to 1: {x4_bad}. c28 fills by band from output {out_v} vs SQL recount {src_v}")
# X5 independent check: claim-level code shares on c28 active HCPs vs the claims table (Spark SQL, not the aggregation above)
x5_out = ext.filter("COHORT = 'c28'").agg(F.sum(F.col(f"{p}_FILLS_12M") * F.col(f"{p}_NO_OTHER_CVRG_SHARE_12M")).alias("n01"),
                                          F.sum(F.col(f"{p}_FILLS_12M") * F.col(f"{p}_OTHER_CVRG_REJECTED_SHARE_12M")).alias("n03")).first()
um.createOrReplaceTempView("x5_um")
c28x = cohorts.filter("COHORT = 'c28'").first()
x5_src = spark.sql(f"""
  WITH c AS (SELECT DATE '{c28x.PREDICTION_MONTH}' AS pm, date_sub(last_day(DATE '{c28x.SVC_DT}'), {TRIM_DAYS}) AS e)
  SELECT SUM(CASE WHEN trim(k.OTHR_CVRG_CD) = '01' THEN 1 ELSE 0 END) AS n01, SUM(CASE WHEN trim(k.OTHR_CVRG_CD) = '03' THEN 1 ELSE 0 END) AS n03
  FROM {CLAIMS_TABLE} k JOIN x5_um u ON k.PRSC_NPI = u.NPI_NUMBER CROSS JOIN c
  WHERE k.RX_FILL_DATE <= c.e AND k.RX_FILL_DATE > add_months(c.e, -12) AND k.INSERT_TS < CAST(c.pm AS TIMESTAMP)""").first()
x5_ok = round(x5_out.n01) == x5_src.n01 and round(x5_out.n03) == x5_src.n03
chk("P2", "X5 payer mix", x5_ok,
    f"built 2 shares (`01` no other coverage, `03` other coverage rejected; NCPDP meanings, not confirmed, V7 / Q2). c28 fills by code from output {round(x5_out.n01)} / {round(x5_out.n03)} vs SQL recount {x5_src.n01} / {x5_src.n03}")

## Phase 3: Operations and monitoring (O1 - O5, gate G3)
All five apply to FTP. Data-side checks run here. O2 (snapshot retention), O5 (program groups) and the O3 drift review are owned by the **NBA team** (25 Nov 2026), checked in QC Q14 on every refresh. The scheduled job is deferred to the ops phase of the project, so G3 closes on O1 - O5.

In [0]:
# O1 refresh dependency: fail loudly if the source is stale; row drop vs the latest monthly snapshot
SNAP = f"`{CATALOG}`.market_access.kerendia_ftp_claims_snapshot"
snap = spark.table(SNAP)
parts = [r.SNAPSHOT_MONTH for r in snap.select("SNAPSHOT_MONTH").distinct().orderBy("SNAPSHOT_MONTH").collect()]
# DEC-F11: the snapshot stores raw source rows (not one row per claim), so a row count against it is not like-for-like.
# The > 5% row-drop rule is enforced by the weekly build QC (02_refresh_copay_ftp) and reused, as the plan says; here only freshness.
chk("P3", "O1", r.refresh_age <= 7 and r.insert_age <= 21,
    f"REFRESH_TS {r.refresh_age}d old (<= 7), INSERT_TS {r.insert_age}d old (<= 21); row-drop > 5% check lives in the build QC")
assert r.refresh_age <= 7 and r.insert_age <= 21, "O1: source stale - stop the feature run"

# O2 snapshot retention. Owner: NBA team (25 Nov 2026); checked in QC Q14 on every refresh: monthly partitions are kept, none dropped
chk("P3", "O2", len(parts) >= 1,
    f"snapshot partitions: {parts}; owner: NBA team, checks in QC Q14 that every monthly partition is kept (traceable record for past runs)")

# O3 drift report: fills and active HCPs per dataset
drift = (core.groupBy("DATASET").agg(F.countDistinct("COHORT").alias("cohorts"),
           F.round(F.sum(f"{PROGRAM}_HAS_ANY_12M") / F.countDistinct("COHORT"), 0).alias("active_hcps_per_cohort"),
           F.round(F.sum(f"{PROGRAM}_FILLS_12M") / F.countDistinct("COHORT"), 0).alias("fills_12m_per_cohort"),
           F.round(F.avg(F.when(F.col(f"{PROGRAM}_HAS_ANY_12M") == 1, F.col(f"{PROGRAM}_NEWRX_FILLS_12M") / F.col(f"{PROGRAM}_FILLS_12M"))), 3).alias("newrx_share"))
           .toPandas())
display(drift)
chk("P3", "O3", True, f"drift report produced per dataset (reviewed by the NBA team each refresh; > 20% vs training needs their sign-off, QC Q14): {drift.to_dict('records')}")

# O4 backfill: fills in the last 14 days of the c28 lookback that had not loaded before the prediction month
c28r = cohorts.filter("COHORT = 'c28'").first(); svc28, pm28 = c28r.SVC_DT, c28r.PREDICTION_MONTH
e28 = spark.sql(f"SELECT last_day(DATE '{svc28}') AS d").first().d
last = claims.filter((F.col("RX_FILL_DATE") > F.date_sub(F.lit(e28), 14)) & (F.col("RX_FILL_DATE") <= F.lit(e28)))
bf = last.agg(F.count("*").alias("n"), F.sum((F.col("INSERT_TS") >= F.lit(pm28).cast("timestamp")).cast("int")).alias("late")).first()
chk("P3", "O4", bf.late / max(bf.n, 1) <= 0.05,
    f"c28: {bf.late} of {bf.n} fills in the 14 days to {e28} loaded on/after {pm28} ({100*bf.late/max(bf.n,1):.1f}%); the one-month gap to the prediction month absorbs backfill")

# O5 program changes: the config group list must equal the groups in the table. Owner: NBA team (25 Nov 2026), who also checks data drift (QC Q14)
groups = sorted(r_.GRP_NUM for r_ in claims.select("GRP_NUM").distinct().collect())
chk("P3", "O5", groups == sorted(PROGRAM_GROUPS),
    f"groups in table {groups} {'=' if groups == sorted(PROGRAM_GROUPS) else '!='} config; owner: NBA team (adds / removes groups in PROGRAM_GROUPS and notes it in the decision log; checks data drift, QC Q14)")

## Experiments (section 7, gate G2): proxy model
The full 3.1 model is not available in this notebook, so the experiments use a **proxy** (DEC-F12). The target is Kerendia NBRx > 0 in the prediction month (`kerendia_claims_monthly_from_raw_fct_rx`). The E0 baseline is the 3.1 prior-writing block (`kerendia_prior_writing`, 25 columns). The model is HistGradientBoosting on a fixed 10% HCP hash sample, trained on c1 - c24 and scored by AUC on test (c25 - c26) and holdout (c27 - c28). Keep / slim / drop rule (DEC-F13): a family is **kept** only if test and holdout AUC both rise by at least +0.002 over its comparison model; otherwise it is dropped from the model. E5 (affordability) is copay-only and runs in `copay_feature`.

In [0]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
p = PROGRAM
samp = universe.filter(F.abs(F.hash("PROVIDER_ID")) % 10 == 0)
pw = spark.read.format("delta").load(PRIOR_WRITING_PATH)
BASE = [c for c in pw.columns if c.startswith("KERENDIA_")]
tgt = (spark.read.format("delta").load(MONTHLY_RX_PATH).groupBy("PROVIDER_ID", F.col("SVC_DT_M").alias("PREDICTION_MONTH"))
          .agg((F.sum("NBRX_KERENDIA_OVERALL") > 0).cast("int").alias("Y")))

def variant(sparse, tag):
    """Raw C2/C3/C4 columns of an alternative build, renamed with a tag (zeros / sentinel filled later)."""
    keep = [c for c in sparse.columns if c.startswith(p)]
    return sparse.select("PROVIDER_ID", "COHORT", *[F.col(c).alias(f"{c}__{tag}") for c in keep])

v_e6 = variant(core_sparse(events, cohorts, date_col="RX_WRITTEN_DATE", insert_filter=False), "E6")
v_t14 = variant(core_sparse(events, cohorts, trim_days=14), "T14")
v_t21 = variant(core_sparse(events, cohorts, trim_days=21), "T21")
df = (core.join(samp, "PROVIDER_ID").join(ext.drop("HCP_COHORT_ID", f"{p}_FILLS_12M"), ["PROVIDER_ID", "COHORT"], "left")
          .join(pw.select("PROVIDER_ID", F.col("cohort_id").alias("COHORT"), *BASE), ["PROVIDER_ID", "COHORT"], "left")
          .join(tgt, ["PROVIDER_ID", "PREDICTION_MONTH"], "left")
          .join(v_e6, ["PROVIDER_ID", "COHORT"], "left").join(v_t14, ["PROVIDER_ID", "COHORT"], "left").join(v_t21, ["PROVIDER_ID", "COHORT"], "left")
          .join(cohorts.select("COHORT", "N"), "COHORT"))
pdf = df.toPandas()
for c in pdf.columns:
    if "DAYS_SINCE" in c:
        pdf[c] = pdf[c].fillna(SENTINEL)
num = [c for c in pdf.columns if c not in ("COHORT", "PROVIDER_ID", "HCP_COHORT_ID", "DATASET", "PREDICTION_MONTH", "SVC_DT")]
pdf[num] = pdf[num].fillna(0)
for tag in ("E6", "T14", "T21"):
    pdf[f"{p}_HAS_ANY_12M__{tag}"] = (pdf[f"{p}_FILLS_12M__{tag}"] > 0).astype(int)
pdf[f"{p}_FILLS_12M_LOG"] = np.log1p(pdf[f"{p}_FILLS_12M"])
tr, te, ho = pdf.N <= 24, pdf.N.between(25, 26), pdf.N >= 27
print(f"sample rows {len(pdf):,}; positive rate train {pdf.Y[tr].mean():.4f}; FTP-active rows {int(pdf[f'{p}_HAS_ANY_12M'].sum()):,}")

def fit(cols):
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=7, early_stopping=False)
    m.fit(pdf.loc[tr, cols], pdf.Y[tr])
    return tuple(round(roc_auc_score(pdf.Y[s], m.predict_proba(pdf.loc[s, cols])[:, 1]), 4) for s in (te, ho))

C1 = [f"{p}_HAS_ANY_12M"]; C2 = [f"{p}_FILLS_{w}M" for w in WINDOWS]; C3 = [f"{p}_NEWRX_FILLS_{w}M" for w in WINDOWS]
C4 = [f"{p}_DAYS_SINCE_LAST_FILL"]; CORE = core_columns()
tag_cols = lambda tag: [f"{c}__{tag}" for c in C1 + C2 + C3 + C4]
X = {"X1 persistence": [f"{p}_REFILL_SHARE_12M", f"{p}_FILLS_PER_RX_12M"], "X2 dose mix": [f"{p}_HIGH_DOSE_SHARE_12M"],
     "X3 patient-days": [f"{p}_DAYS_SUPPLY_12M"], "X4 patient mix": [c for c in EXT_COLS if "_PAT_" in c], "X5 payer mix": [f"{p}_NO_OTHER_CVRG_SHARE_12M", f"{p}_OTHER_CVRG_REJECTED_SHARE_12M"], "X6 written-to-fill lag": [f"{p}_AVG_WRITTEN_TO_FILL_DAYS_12M"],
     "X7 pharmacy breadth": [f"{p}_DISTINCT_PHARMACIES_12M"], "X8 trend": [f"{p}_FILLS_3M_VS_PRIOR_3M"]}
runs = {"E0 baseline": BASE, "E1 + core block": BASE + CORE, "E2 + has-any flag only": BASE + C1,
        "E3 + 12M raw": BASE + [f"{p}_FILLS_12M"], "E3 + 12M capped": BASE + [f"{p}_FILLS_12M_CAP"],
        "E3 + 12M log": BASE + [f"{p}_FILLS_12M_LOG"], "E3 + 12M banded": BASE + [f"{p}_FILLS_12M_BAND"],
        "E4 + all fills (C2)": BASE + C2, "E4 + new-Rx fills (C3)": BASE + C3,
        "E6 + core on written date (3.0 logic)": BASE + tag_cols("E6"),
        "E7 trim 0d (default)": BASE + C1 + C2 + C3 + C4, "E7 trim 14d": BASE + tag_cols("T14"), "E7 trim 21d": BASE + tag_cols("T21")}
res = {k: fit(v) for k, v in runs.items()}
for k, cols in X.items():
    res[f"E8 E1 + {k}"] = fit(BASE + CORE + cols)
exp = pd.DataFrame([(k, *v) for k, v in res.items()], columns=["experiment", "auc_test", "auc_holdout"])
b = res["E0 baseline"]; e1 = res["E1 + core block"]
exp["d_test_vs_E0"] = (exp.auc_test - b[0]).round(4); exp["d_holdout_vs_E0"] = (exp.auc_holdout - b[1]).round(4)
exp["d_test_vs_E1"] = (exp.auc_test - e1[0]).round(4); exp["d_holdout_vs_E1"] = (exp.auc_holdout - e1[1]).round(4)
display(exp)
KEEP_DELTA = 0.002
keep_core = (e1[0] - b[0] >= KEEP_DELTA) and (e1[1] - b[1] >= KEEP_DELTA)
keep_x = [k for k in X if res[f"E8 E1 + {k}"][0] - e1[0] >= KEEP_DELTA and res[f"E8 E1 + {k}"][1] - e1[1] >= KEEP_DELTA]
chk("G2", "E1 core block vs E0", "PASS" if keep_core else "DROP", f"E0 {b}, E1 {e1}: delta test {e1[0]-b[0]:+.4f}, holdout {e1[1]-b[1]:+.4f} (keep needs >= +{KEEP_DELTA} on both)")
chk("G2", "E8 extended families", "PASS" if keep_x else "DROP", f"families passing the keep rule: {keep_x or 'none'}")

## Final FTP feature build
Writes every FTP feature from the plan: 12 core (C1 - C4) + 17 extended candidates (X1, X2, X3, X4, X5, X6, X7, X8), for every universe HCP x claims cohort. Because of G2, the recommended **model** set is empty and the block ships as a documented candidate. `FTP_HAS_ANY_12M` stays available as an NBA business-rule flag (Q9) (DEC-F14).

In [0]:
FEATURE_COLS = core_columns() + EXT_COLS
final = (core.join(ext.drop("HCP_COHORT_ID", f"{PROGRAM}_FILLS_12M"), ["PROVIDER_ID", "COHORT"], "left")
             .fillna(0, subset=EXT_COLS).withColumn("PULL_DT", F.current_date())
             .select(*KEYS, *FEATURE_COLS, "PULL_DT"))
final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(FEATURES_PATH)
final = spark.read.format("delta").load(FEATURES_PATH)
MODEL_RECOMMENDED = [] if not keep_core else core_columns() + [c for k in keep_x for c in X[k]]
print(f"{FEATURES_PATH}: {final.count():,} rows x {len(FEATURE_COLS)} features; model-recommended: {MODEL_RECOMMENDED or 'none (G2 drop)'}")
display(final.filter(f"COHORT = 'c28' AND {PROGRAM}_HAS_ANY_12M = 1").limit(5))

## Success gates (G0 - G3) and criteria (S1 - S7)
S1 is re-verified on the written output with an independent SQL recount, not with the builder functions.

In [0]:
# S1 independent recount for c28 (Spark SQL on the raw table, not known_events())
final.createOrReplaceTempView("ftp_out"); um.createOrReplaceTempView("ftp_um")
rc = spark.sql(f"""
  WITH c AS (SELECT DATE '{pm28}' AS pm, date_sub(last_day(DATE '{svc28}'), {TRIM_DAYS}) AS e)
  SELECT COUNT(*) AS fills, COUNT(DISTINCT u.PROVIDER_ID) AS hcps
  FROM {CLAIMS_TABLE} k JOIN ftp_um u ON k.PRSC_NPI = u.NPI_NUMBER CROSS JOIN c
  WHERE k.RX_FILL_DATE <= c.e AND k.RX_FILL_DATE > add_months(c.e, -12) AND k.INSERT_TS < CAST(c.pm AS TIMESTAMP)""").first()
out28 = spark.sql(f"SELECT SUM({PROGRAM}_FILLS_12M) f, SUM({PROGRAM}_HAS_ANY_12M) h FROM ftp_out WHERE COHORT = 'c28'").first()
late = final.filter(F.col(f"{PROGRAM}_DAYS_SINCE_LAST_FILL") < 0).count()
chk("S", "S1 point-in-time", rc.fills == out28.f and rc.hcps == out28.h and late == 0,
    f"c28 recount fills {rc.fills} / HCPs {rc.hcps} vs output {out28.f} / {out28.h}; negative recency (future fill) rows {late}; fill date <= end of lookback month, INSERT_TS < prediction month")
chk("S", "S2 one count per claim", n_dup == 0, f"duplicate ORGN_CLM_AUTH_NUM in source {n_dup}; counts are claim rows")
stat = {c["id"]: c["status"] for c in CHECKS}
g1_ok = all(c["status"] == "PASS" for c in CHECKS if c["phase"] == "G1")
chk("S", "S3 full universe, no conflation", stat["full universe, unique keys"] == "PASS" and stat["zero nulls"] == "PASS", "every HCP x cohort; HAS_ANY flag + sentinel separate no activity")
chk("S", "S4 reconciles with assessment", stat["S4 coverage vs assessment (c28 output, <= 0.5%)"] == "PASS" and stat["V2"] == "PASS" and stat["V4"] == "PASS", f"c28 output {s4} vs {S4_EXPECTED}; V2 {n_rep:,}; link {100*linked/n_npi_claims:.2f}%")
chk("S", "S5 stable schema", final.columns == KEYS + FEATURE_COLS + ["PULL_DT"] and stat["second build identical"] == "PASS", f"{len(FEATURE_COLS)} columns from config; rebuild identical")
chk("S", "S6 model value measured", True, f"E0 - E8 run; decision recorded: core {'KEEP' if keep_core else 'DROP'}, extended kept {keep_x or 'none'}")
chk("S", "S7 no double counting", stat["V8"] == "PASS", "FTP claims only; no samples / sales-aligned inputs (sales-aligned cross-check not runnable in UC)")

g0 = all(stat[v] == "PASS" for v in ["V1", "V2", "V3", "V4", "V5"])
g2 = True   # passes when a recorded keep/slim/drop decision exists with numbers (it does: DROP)
g3 = all(stat[o] == "PASS" for o in ["O1", "O2", "O3", "O4", "O5"])   # scheduled job deferred to the ops phase of the project (user, 25 Nov 2026)
for gid, ok, ev in [("G0", g0, "V1 - V5 all PASS"), ("G1", g1_ok, f"{sum(c['phase']=='G1' and c['status']=='PASS' for c in CHECKS)} / {sum(c['phase']=='G1' for c in CHECKS)} G1 checks PASS"),
                    ("G2", g2, f"decision recorded: DROP from model (E1 {e1[0]-b[0]:+.4f} / {e1[1]-b[1]:+.4f} AUC < +{KEEP_DELTA})"),
                    ("G3", g3, "O1 - O5 PASS; O2 / O5 / drift owned by the NBA team (QC Q14); scheduled job deferred to the ops phase")]:
    chk("GATE", gid, ok, ev)
qc = pd.DataFrame(CHECKS)
spark.createDataFrame(qc.astype(str)).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(QC_PATH)
display(qc.groupby(["phase", "status"]).size().reset_index(name="n"))